## Creates patterns and semantic_annotations tables

### all annotated verb patterns

In [1]:
import sqlite3
import pandas as pd
from estnltk import Text
import sys
sys.path.append("..")
from common_display import display_db_table 

## Configuration

In [2]:
DB_DIR = "../example_data"
SOURCE_DIR = "../source_data"

# annoteeritud andmed
ANNOTATION_FILE = f"{SOURCE_DIR}/every_verb_case_obl.csv"

VERB_PATTERN_DB = f"{DB_DIR}/verb_patterns.db"

# patterns lõpptabel
PATTERN_TABLE_NAME = "patterns" 
# semnatiliste annotatsioonide tabel
SEMANTIC_ANNOTATIONS_TABLE = "semantic_annotations"

# deprel mustri tabelisse
DEPREL = 'obl'

## Create verb_patterns database

In [3]:
con = sqlite3.connect(VERB_PATTERN_DB)
cur = con.cursor()

## Workflow

### Read in and process verb annotations

In [3]:
margendused = pd.read_csv(ANNOTATION_FILE, sep=";", encoding="utf-8")
margendused

,verbobl,verb,case,isikumäärus,aja-kohamäärus,muu
0,saama - abl (kellelt/millelt),saama,abl (kellelt/millelt),vahel,vahel,mitte kunagi
1,tulema - abl (kellelt/millelt),tulema,abl (kellelt/millelt),vahel,vahel,mitte kunagi
2,küsima - abl (kellelt/millelt),küsima,abl (kellelt/millelt),alati,mitte kunagi,mitte kunagi
3,nõudma - abl (kellelt/millelt),nõudma,abl (kellelt/millelt),alati,mitte kunagi,mitte kunagi
4,võtma - abl (kellelt/millelt),võtma,abl (kellelt/millelt),vahel,vahel,mitte kunagi
...,...,...,...,...,...,...
10574,musitseerima - in (kelles/milles),musitseerima,in (kelles/milles),mitte kunagi,alati,muu
10575,kõigutama - in (kelles/milles),kõigutama,in (kelles/milles),mitte kunagi,mitte kunagi,muu
10576,kätlema - in (kelles/milles),kätlema,in (kelles/milles),mitte kunagi,alati,muu
10577,kõmmutama - in (kelles/milles),kõmmutama,in (kelles/milles),mitte kunagi,alati,mitte kunagi


#### process_data(deprel)

- võtab sisse:
    - annotastioonide dataframe-i
    - deprel
- vajadusel tekitab ühest kirjest kaks kui verbifraasisi on komaga eraldatud kaks kaassõna
     - nt 'olema kokku, võlgu' -> 'olema kokku' ja 'olema võlgu'
- väljastab:
    - dataframe-i kus ei ole mitme kaassõnaga verbe

In [4]:
def process_data(annotations, deprel):
    annotations = annotations.reset_index().drop("index", axis=1)

    df_for_verb_processing = []
    for i in range(len(annotations)):
        verb = annotations.iloc[i]["verb"].strip()
        case_osad =  annotations.iloc[i]["case"].strip().split(" ")
        if len(case_osad)>=2:
            isik = annotations.iloc[i]["isikumäärus"]
            koht = annotations.iloc[i]["aja-kohamäärus"]
            muu = annotations.iloc[i]["muu"]
            case = case_osad[0].strip()
            gov = case_osad[1].replace("(", "").replace(")", "").strip()
            pat = verb + " " + gov
            if "," in verb:
                osad = verb.split(" ")
                v = osad[0].strip()
                v1 = osad[1].replace(",", "").strip()
                v2 = osad[2].replace(",", "").strip()
                # hoiatus, et verbis on rohkem kui 2 kaassõna ning koodi tuleks muuta
                if len(osad)>3:
                    print("rohkem osasid!")
                df_for_verb_processing.append((pat, v+" "+v1, case, isik, koht, muu))
                df_for_verb_processing.append((pat, v+" "+v2, case, isik, koht, muu))
            else:
                df_for_verb_processing.append((pat, verb, case, isik, koht, muu))

    df_step1 = pd.DataFrame(df_for_verb_processing, columns=["pattern", "verb_phrase", "phrase_case", "isik", "koht", "muu"])
    
    return df_step1

In [5]:
corrected_verbs = process_data(margendused, DEPREL)
corrected_verbs

,pattern,verb_phrase,phrase_case,isik,koht,muu
0,saama kellelt/millelt,saama,abl,vahel,vahel,mitte kunagi
1,tulema kellelt/millelt,tulema,abl,vahel,vahel,mitte kunagi
2,küsima kellelt/millelt,küsima,abl,alati,mitte kunagi,mitte kunagi
3,nõudma kellelt/millelt,nõudma,abl,alati,mitte kunagi,mitte kunagi
4,võtma kellelt/millelt,võtma,abl,vahel,vahel,mitte kunagi
...,...,...,...,...,...,...
10600,musitseerima kelles/milles,musitseerima,in,mitte kunagi,alati,muu
10601,kõigutama kelles/milles,kõigutama,in,mitte kunagi,mitte kunagi,muu
10602,kätlema kelles/milles,kätlema,in,mitte kunagi,alati,muu
10603,kõmmutama kelles/milles,kõmmutama,in,mitte kunagi,alati,mitte kunagi


#### process_verbs(df)

- võtab sisse:
    - mustrite/verbide dataframe-i (eelduseks veeru 'verb_phrase' olemasolu, mis sisaldab kogu verbifraasi)
- lööb verbifraasi lahku peaverbiks ja komponentideks
- oletus on, et pikema konstruktsiooni viimane verbist liige on peaverb
- eelduslikult on verbis 1 compound (maksimaalselt 3 compoundi kui lubada välja kommenteeritud read)
- väljastab: 
    - verbi peasõnade listi
    - verbi komponendi listi

In [6]:
def process_verbs(df):
    verb_word = []
    compound_prt1 = []
    #compound_prt2 = []
    #compound_prt3 = []

    for idx, row in df.iterrows():
        verb = ''
        compound = ['', '', ''] # max 3 compound pieces
        pieces = row['verb_phrase'].strip().split()
        
        # (pea)verbi leidmine. 
        j = len(pieces)-1 
        while j >= 0:
            text = Text(pieces[j]).tag_layer('morph_analysis')
            if 'V' in text.morph_analysis.partofspeech[0]:
                verb = pieces[j]
                pieces.pop(j)
                break
            j-=1
        # allesjäänud jupid määratakse konstruktsiooni ülejäänud osadeks
        for i in range(len(pieces)): 
            compound[i] = pieces[i]

        verb_word.append(verb)
        compound_prt1.append(compound[0])
        #compound_prt2.append(compound[1])
        #compound_prt3.append(compound[2])
        
    return verb_word, compound_prt1 #, compound_prt2, compound_prt3

In [7]:
verbs_list, comp_list1 = process_verbs(corrected_verbs)

In [23]:
print(verbs_list[:10])
print(comp_list1[:10])

['saama', 'tulema', 'küsima', 'nõudma', 'võtma', 'ootama', 'leidma', 'ostma', 'pärinema', 'paluma']
['', '', '', '', '', '', '', '', '', '']


#### create_patterns_dataframe(df, verbs, compunds, deprel)

- võtab sisse:
    - dataframe-i, kus on parandatud verbifraasid
    - verbi peasõnade listi
    - verbi compound listi
    - deprel
    - vajadusel saab anda sisse rohkem compound liste 
- kutsub välja process_verbs
- lisab tabelisse verbi komponendid ja muu vajaliku info
- väljastab:
    - dataframe-i, kus on vajalik info patterns tabeli jaoks

In [11]:
def create_patterns_dataframe(df1, verb_list, compund_list, deprel):
    df1['verb_word'] = verb_list
    df1['verb_compound'] = compund_list
    #df1['compound_prt2'] = comp_list2
    #df1['compound_prt3'] = comp_list3
    df1['phrase_nr'] = 1
    df1['adp'] = ''
    df1['deprel'] = deprel
    df1['inf_verb'] = ''
    df1.insert(0, 'pat_id', range(1, 1 + len(df1)))
    
    # drop unneccesary column
    df2 = df1.drop(columns=['verb_phrase'])

    return df2

In [12]:
df = create_patterns_dataframe(corrected_verbs, verbs_list, comp_list1, DEPREL)
df

,pat_id,pattern,phrase_case,isik,koht,muu,verb_word,verb_compound,phrase_nr,adp,deprel,inf_verb
0,1,saama kellelt/millelt,abl,vahel,vahel,mitte kunagi,saama,,1,,obl,
1,2,tulema kellelt/millelt,abl,vahel,vahel,mitte kunagi,tulema,,1,,obl,
2,3,küsima kellelt/millelt,abl,alati,mitte kunagi,mitte kunagi,küsima,,1,,obl,
3,4,nõudma kellelt/millelt,abl,alati,mitte kunagi,mitte kunagi,nõudma,,1,,obl,
4,5,võtma kellelt/millelt,abl,vahel,vahel,mitte kunagi,võtma,,1,,obl,
...,...,...,...,...,...,...,...,...,...,...,...,...
10600,10601,musitseerima kelles/milles,in,mitte kunagi,alati,muu,musitseerima,,1,,obl,
10601,10602,kõigutama kelles/milles,in,mitte kunagi,mitte kunagi,muu,kõigutama,,1,,obl,
10602,10603,kätlema kelles/milles,in,mitte kunagi,alati,muu,kätlema,,1,,obl,
10603,10604,kõmmutama kelles/milles,in,mitte kunagi,alati,mitte kunagi,kõmmutama,,1,,obl,


In [13]:
df.to_csv(f"{SOURCE_DIR}/verb_patterns.csv", sep=",", encoding="utf-8", index = False)
df = pd.read_csv(f"{SOURCE_DIR}/verb_patterns.csv", sep=",", encoding="utf-8")
df = df[~df["verb_word"].isna()]
df = df.fillna('')

### Create patterns table in database

In [23]:
cur.execute("""DROP TABLE IF EXISTS {name}""".format(name=PATTERN_TABLE_NAME))

cur.execute(
    """CREATE TABLE {tablename}
    (pat_id INTEGER PRIMARY KEY, pattern TEXT, phrase_case TEXT, verb_word TEXT, verb_compound TEXT, phrase_nr INT, adp TEXT, deprel TEXT, inf_verb TEXT)
    """.format(tablename=PATTERN_TABLE_NAME)
)


insert = f"INSERT INTO {PATTERN_TABLE_NAME} "
for idx, row in df.iterrows():
    cur.execute(insert + """
    (pat_id, pattern, phrase_case, verb_word, verb_compound, phrase_nr, adp, deprel, inf_verb) 
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?);""", 
    (row["pat_id"], row['pattern'], row['phrase_case'], row['verb_word'], row['verb_compound'], row['phrase_nr'], row['adp'], row['deprel'], row['inf_verb']))
    
    con.commit()

### Create semantic_annotations table in database

In [40]:
cur.execute("""DROP TABLE IF EXISTS {name}""".format(name=SEMANTIC_ANNOTATIONS_TABLE))

cur.execute(
    """CREATE TABLE {tablename}
    (pattern_id INT,phrase_nr INT, semantic_role TEXT, certainty TEXT)
    """.format(tablename=SEMANTIC_ANNOTATIONS_TABLE)
)


for idx, row in df.iterrows():

    pat_id = row["pat_id"]
    phrase_nr = row["phrase_nr"]
    
    for role in ["isik", "koht", "muu"]:
        certain = row[role]

        insert = f"INSERT INTO {SEMANTIC_ANNOTATIONS_TABLE} "
        cur.execute(insert + """
            (pattern_id, phrase_nr, semantic_role, certainty) 
            VALUES (?, ?, ?, ?);""", 
            (pat_id, phrase_nr, role, certain)
        )
    
        con.commit()

### Check table contents

In [18]:
query = """SELECT count(*) as cnt from {new_table}""".format(new_table = PATTERN_TABLE_NAME)
r1 = pd.read_sql_query(query, con)
assert r1["cnt"][0] == 10515, "mustrite tabelis ei ole eelduslik arv ridu"

query = """SELECT count(*) as cnt from {new_table}""".format(new_table = SEMANTIC_ANNOTATIONS_TABLE)
r2 = pd.read_sql_query(query, con)
assert r2["cnt"][0] == 31545, "semantilise rolli tabelis ei ole eelduslik arv ridu"

Kontroll, et andmed on tabelis soovitud kujul ja veerud ei ole nihkes

In [4]:
display_db_table(con, PATTERN_TABLE_NAME, 5, 'head')

,pat_id,pattern,phrase_case,verb_word,verb_compound,phrase_nr,adp,deprel,inf_verb
0,1,saama kellelt/millelt,abl,saama,,1,,obl,
1,2,tulema kellelt/millelt,abl,tulema,,1,,obl,
2,3,küsima kellelt/millelt,abl,küsima,,1,,obl,
3,4,nõudma kellelt/millelt,abl,nõudma,,1,,obl,
4,5,võtma kellelt/millelt,abl,võtma,,1,,obl,


In [5]:
display_db_table(con, SEMANTIC_ANNOTATIONS_TABLE, 5, 'head')

,pattern_id,phrase_nr,semantic_role,certainty
0,1,1,isik,vahel
1,1,1,koht,vahel
2,1,1,muu,mitte kunagi
3,2,1,isik,vahel
4,2,1,koht,vahel


In [6]:
con.close()